In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TripletLoss(nn.Module):
    """
    Triplet Loss: L = max(d(a,p) - d(a,n) + alpha, 0)
    Where:
    - a: anchor
    - p: positive (same person)
    - n: negative (different person)
    """
    def __init__(self, margin=1.0):
        super(TripletLoss, self).__init__()
        self.margin = margin
    
    def forward(self, anchor, positive, negative):
        # Calculate distances
        pos_dist = torch.norm(anchor - positive, p=2, dim=1)
        neg_dist = torch.norm(anchor - negative, p=2, dim=1)
        
        # Triplet loss
        loss = torch.relu(pos_dist - neg_dist + self.margin)
        return loss.mean()

class FaceEmbeddingNet(nn.Module):
    def __init__(self, embedding_dim=128):
        super(FaceEmbeddingNet, self).__init__()
        
        # Simple CNN for face embedding
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=5, stride=2, padding=2)
        self.bn2 = nn.BatchNorm2d(128)
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(256)
        self.pool3 = nn.AdaptiveAvgPool2d(1)
        
        self.fc = nn.Linear(256, embedding_dim)
    
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        # L2 normalize embeddings
        x = F.normalize(x, p=2, dim=1)
        return x

# Create model
face_net = FaceEmbeddingNet(embedding_dim=128).to(device)
triplet_loss = TripletLoss(margin=1.0)

print("Face embedding network created")
print(f"Parameters: {sum(p.numel() for p in face_net.parameters()):,}")

## 1. Triplet Loss for Face Recognition

# Face Detection and Recognition
## Detecting and Recognizing Human Faces

Topics covered:
- Face detection (MTCNN, RetinaFace, SSD)
- Face alignment and landmark detection
- Face recognition with embeddings
- Triplet loss and metric learning
- FaceNet, ArcFace, CosFace
- Face verification and identification
- Face clustering
- Privacy and ethical considerations